# Домашнее задание

**Задача:** реализовать и запустить модель LDA и Gibbs Sampling с числом тегов 20. Вывести топ-10 слов по каждому тегу.

In [3]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer

print("Загрузка данных...")
data = fetch_20newsgroups(subset="train", remove=("headers", "footers", "quotes"))
texts = data.data[:2000]
y_true = data.target[:2000]
label_names = data.target_names

print("Векторизация...")
vectorizer = CountVectorizer(
    max_features=3000,
    stop_words="english",
    min_df=8,
    max_df=0.6
)
X = vectorizer.fit_transform(texts)
vocab = vectorizer.get_feature_names_out()
M, V = X.shape
print(f"Документов: {M}, словарь: {V}")

print("Разворачивание в токены...")
all_words = []
all_docs = []
for d in range(M):
    row = X[d]
    for w, c in zip(row.indices, row.data):
        c = int(c)
        all_words.extend([w] * c)
        all_docs.extend([d] * c)

all_words = np.asarray(all_words, dtype=np.int32)
all_docs = np.asarray(all_docs, dtype=np.int32)
W = len(all_words)
print(f"Всего токенов: {W}")

K = 20
alpha = 0.1
beta = 0.01
n_iter = 100
beta_sum = beta * V
alpha_vec = np.full(K, alpha, dtype=np.float64)

n_dk = np.zeros((M, K), dtype=np.int32)
n_kw = np.zeros((K, V), dtype=np.int32)
n_k = np.zeros(K, dtype=np.int32)

rng = np.random.default_rng(0)
z = rng.integers(0, K, size=W, dtype=np.int32)

for i in range(W):
    d = all_docs[i]
    w = all_words[i]
    k = z[i]
    n_dk[d, k] += 1
    n_kw[k, w] += 1
    n_k[k] += 1

print("Gibbs Sampling...")
for it in range(n_iter):
    print(f"Итерация {it+1}/{n_iter}")
    for i in rng.permutation(W):
        d = all_docs[i]
        w = all_words[i]
        k_old = z[i]

        n_dk[d, k_old] -= 1
        n_kw[k_old, w] -= 1
        n_k[k_old] -= 1

        p = (n_dk[d].astype(np.float64) + alpha_vec) * (n_kw[:, w].astype(np.float64) + beta) / (n_k.astype(np.float64) + beta_sum)
        s = p.sum()
        if s > 0:
            p /= s
        else:
            p[:] = 1.0 / K

        k_new = rng.choice(K, p=p)
        z[i] = k_new

        n_dk[d, k_new] += 1
        n_kw[k_new, w] += 1
        n_k[k_new] += 1

phi = (n_kw + beta) / (n_k[:, None] + beta_sum)
theta = (n_dk + alpha) / (n_dk.sum(axis=1, keepdims=True) + K * alpha)

print("\nТоп-10 слов по темам:")
for k in range(K):
    top_idx = np.argsort(phi[k])[-10:][::-1]
    top_words = [vocab[i] for i in top_idx]
    print(f"Тема {k:2d}: {', '.join(top_words)}")

print("\nСопоставление тем с категориями датасета:")
top_docs = 50
for k in range(K):
    best_docs = np.argsort(theta[:, k])[::-1][:top_docs]
    counts = np.bincount(y_true[best_docs], minlength=len(label_names))
    best_label = counts.argmax()
    share = counts[best_label] / top_docs
    print(f"Тема {k:2d} -> {label_names[best_label]} ({share:.2f})")


Загрузка данных...
Векторизация...
Документов: 2000, словарь: 3000
Разворачивание в токены...
Всего токенов: 117985
Gibbs Sampling...
Итерация 1/100
Итерация 2/100
Итерация 3/100
Итерация 4/100
Итерация 5/100
Итерация 6/100
Итерация 7/100
Итерация 8/100
Итерация 9/100
Итерация 10/100
Итерация 11/100
Итерация 12/100
Итерация 13/100
Итерация 14/100
Итерация 15/100
Итерация 16/100
Итерация 17/100
Итерация 18/100
Итерация 19/100
Итерация 20/100
Итерация 21/100
Итерация 22/100
Итерация 23/100
Итерация 24/100
Итерация 25/100
Итерация 26/100
Итерация 27/100
Итерация 28/100
Итерация 29/100
Итерация 30/100
Итерация 31/100
Итерация 32/100
Итерация 33/100
Итерация 34/100
Итерация 35/100
Итерация 36/100
Итерация 37/100
Итерация 38/100
Итерация 39/100
Итерация 40/100
Итерация 41/100
Итерация 42/100
Итерация 43/100
Итерация 44/100
Итерация 45/100
Итерация 46/100
Итерация 47/100
Итерация 48/100
Итерация 49/100
Итерация 50/100
Итерация 51/100
Итерация 52/100
Итерация 53/100
Итерация 54/100
Итерация 55